In [19]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


url = "https://www.vegasinsider.com/college-basketball/odds/las-vegas/"

In [20]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


BASE_URL = "https://www.vegasinsider.com/college-basketball/odds/las-vegas/"
DATE = "2025-11-21"

def _get_soup(date_str: str) -> BeautifulSoup:
    params = {"date": date_str}
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        )
    }
    resp = requests.get(BASE_URL, params=params, headers=headers)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "lxml")


def _parse_market(soup: BeautifulSoup, market: str) -> pd.DataFrame:
    """
    market: 'spread' or 'total'
    - 'spread' -> away/home spreads
    - 'total'  -> away row = Over, home row = Under
    """
    assert market in {"spread", "total"}

    # each market has tbody id like: odds-table-spread--0, odds-table-total--0, ...
    bodies = soup.select(f'tbody[id^="odds-table-{market}--"]')

    all_games = []

    for tbody in bodies:
        # header row above tbody tells us the books (Open, bet365, betmgm, draftkings, hardrock, Consensus, ...)
        thead = tbody.find_previous("thead")
        header_row = thead.find("tr")
        ths = header_row.find_all("th")[1:]  # skip the game-time column

        books = []
        for th in ths:
            img = th.find("img")
            if img and img.get("alt"):
                name = img["alt"].strip()
            else:
                txt = th.get_text(strip=True)
                name = txt if txt else "Consensus"
            # normalize to something simple for column names
            books.append(name.lower().replace("&", "and").replace(" ", "_"))

        def parse_odds_row(tr):
            """Return list of (value, price) for each book column."""
            cells = tr.find_all("td", class_="game-odds")[: len(books)]
            out = []
            for c in cells:
                value_tag = c.find("span", class_="data-value")
                price_tag = c.find("small", class_="data-odds")
                value = value_tag.get_text(strip=True) if value_tag else None
                price = price_tag.get_text(strip=True) if price_tag else None
                if value == "N/A":
                    value = None
                out.append((value, price))
            return out

        rows = tbody.find_all("tr", recursive=False)
        i = 0
        while i < len(rows):
            row = rows[i]

            # look for a time row (starts each game block)
            time_cell = row.find("td", class_="game-time")
            if not time_cell:
                i += 1
                continue

            # game time
            time_span = time_cell.find("span", attrs={"data-role": "localtime"})
            game_time = (
                time_span["data-value"]
                if time_span and time_span.has_attr("data-value")
                else None
            )

            # structure:
            # row i     -> time + "Open" + book logos
            # row i+1   -> away team row
            # row i+2   -> home team row
            # row i+3   -> matchup links
            away_row = rows[i + 1]
            home_row = rows[i + 2]
            i += 4  # jump to next game block

            # teams
            away_team_tag = away_row.select_one(".team-name")
            home_team_tag = home_row.select_one(".team-name")
            away_team = away_team_tag.get_text(strip=True) if away_team_tag else None
            home_team = home_team_tag.get_text(strip=True) if home_team_tag else None

            # rotation numbers (306563, 306564, etc.)
            def get_rot(tr):
                span = tr.select_one(".team-plate span span")
                return span.get_text(strip=True) if span else None

            away_rot = get_rot(away_row)
            home_rot = get_rot(home_row)

            away_odds = parse_odds_row(away_row)
            home_odds = parse_odds_row(home_row)

            game = {
                "game_time": game_time,
                "away_team": away_team,
                "home_team": home_team,
                "away_rot": away_rot,
                "home_rot": home_rot,
            }

            # first "book" is actually the Open column, the rest are books
            for book_name, (a_val, a_price), (h_val, h_price) in zip(
                books, away_odds, home_odds
            ):
                if market == "spread":
                    game[f"{book_name}_away_spread"] = a_val
                    game[f"{book_name}_away_spread_price"] = a_price
                    game[f"{book_name}_home_spread"] = h_val
                    game[f"{book_name}_home_spread_price"] = h_price
                else:  # total
                    # away row = Over, home row = Under (as in your o140.5 / u140.5 snippet)
                    game[f"{book_name}_total_over"] = a_val
                    game[f"{book_name}_total_over_price"] = a_price
                    game[f"{book_name}_total_under"] = h_val
                    game[f"{book_name}_total_under_price"] = h_price

            all_games.append(game)

    return pd.DataFrame(all_games)


def get_cbb_spread_and_total(date_str: str) -> pd.DataFrame:
    """
    Returns one DataFrame with both spreads and totals (including Open).
    """
    soup = _get_soup(date_str)
    spread_df = _parse_market(soup, "spread")
    total_df = _parse_market(soup, "total")

    # merge on the game identifiers – time + teams is usually enough
    on_cols = ["game_time", "away_team", "home_team", "away_rot", "home_rot"]
    df = pd.merge(spread_df, total_df, on=on_cols, how="outer", suffixes=("_spr", "_tot"))
    return df


if __name__ == "__main__":
    df = get_cbb_spread_and_total(DATE)


In [21]:
df = df[['away_team', 'home_team','open_home_spread', 'bet365_home_spread', 'betmgm_home_spread', 'draftkings_home_spread','hardrock_home_spread','open_total_over','bet365_total_over','betmgm_total_over','draftkings_total_over','hardrock_total_over']]

In [22]:
import os
os.makedirs(f"data/odds/{DATE}/", exist_ok=True)

df.to_csv(f"data/odds/{DATE}/odds.csv")

In [23]:
df.isna().sum()

away_team                  0
home_team                  0
open_home_spread          65
bet365_home_spread        65
betmgm_home_spread        65
draftkings_home_spread    65
hardrock_home_spread      65
open_total_over           65
bet365_total_over         65
betmgm_total_over         65
draftkings_total_over     65
hardrock_total_over       65
dtype: int64